# DistilBERT fine-tuning (Colab)

Trains `distilbert-base-uncased` with soft-label BCE on the **same** train / valid /
identity hold-out split that `prepare.py` produces for the TF-IDF baseline, so the
only difference from the baseline is the model.

**Before running**
1. *Runtime > Change runtime type > T4 GPU*.
2. In Google Drive create `MyDrive/toxicity-mlops/data/` and upload, from the repo's
   `data/processed/` folder (created by `dvc repro`): `train.csv`, `valid.csv`,
   `identity_eval.parquet`.

Outputs are written to `MyDrive/toxicity-mlops/distilbert_out/`.
Expected time on a T4: roughly 30-40 minutes for 2 epochs.

> Content warning: the dataset contains abusive and offensive text.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/toxicity-mlops'
!ls -la {DRIVE_DIR}/data

In [ ]:
BRANCH = 'main'  # branch of the repository to train from
!rm -rf /content/repo
!git clone --depth 1 --branch {BRANCH} https://github.com/saikatsam11/MLOps-Midterm.git /content/repo
!pip install -q -r /content/repo/requirements-distilbert.txt

In [ ]:
!cd /content/repo && python src/train_distilbert.py     --data-dir {DRIVE_DIR}/data     --out-dir {DRIVE_DIR}/distilbert_out     --max-len 192 --batch-size 32 --epochs 2 --lr 2e-5

In [ ]:
!cat {DRIVE_DIR}/distilbert_out/train_config.json
!ls -la {DRIVE_DIR}/distilbert_out

## Next (on your machine)

Download the whole `distilbert_out/` folder from Drive into the repository root, then:

```bash
python src/evaluate.py  --scores distilbert_out/valid_scores.csv         --out-dir metrics/distilbert
python src/bias_eval.py --scores distilbert_out/identity_eval_scores.csv --out-dir metrics/distilbert
python src/log_mlflow.py --run-name distilbert --metrics-dir metrics/distilbert --external-model distilbert_out
```